Status: exploratory

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import gc

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from scipy import ndimage
from skimage.filters import threshold_multiotsu
from skimage.morphology import remove_small_holes

import bosperrus
from bosperrus.fit import ConstantFit, PiecewiseLinearFit
from bosperrus.distances import distance_to_mask, distance_to_rectangular_border

Image.MAX_IMAGE_PIXELS = None

## Datasets

Three 8µm Visium HD samples, processed identically and compared side by side.

In [3]:
BASE = "/home/woody/iwbn/iwbn007h/visium_hd_demo_data"

DATASETS = [
    {
        "name": "breast_cancer",
        "h5ad": f"{BASE}/Visium_HD_11mm_Human_Breast_Cancer/breast_cancer_008um_filtered.h5ad",
        "binned_dir": f"{BASE}/Visium_HD_11mm_Human_Breast_Cancer/binned_outputs/square_008um",
    },
    {
        "name": "colon_cancer_ff",
        "h5ad": f"{BASE}/Visium_HD_11mm_Human_Colon_Cancer_Fresh_Frozen/colon_cancer_ff_008um_filtered.h5ad",
        "binned_dir": f"{BASE}/Visium_HD_11mm_Human_Colon_Cancer_Fresh_Frozen/binned_outputs/square_008um",
    },
    {
        "name": "pancreas",
        "h5ad": f"{BASE}/Visium_HD_11mm_Human_Pancreas/pancreas_008um_filtered.h5ad",
        "binned_dir": f"{BASE}/Visium_HD_11mm_Human_Pancreas/binned_outputs/square_008um",
    },
]

In [4]:
MAX_HOLE_AREA = 500  # background gaps smaller than this get closed; larger ones (real tissue holes) are preserved
BIN_SIZE_UM = 8.0  # all three datasets are square_008um; adjust if that changes

## Per-dataset analysis

For each sample: load, compute `log1p(total_counts)`, build the tissue mask
(multi-Otsu, 3 classes — background/tissue/nuclei — cleaned with
`binary_fill_holes`+`binary_opening`), then fit `log1p_total_counts` against
two independent distances over all tissue bins — intrinsic (distance to
background via the segmentation mask) and extrinsic (distance to the
capture-area's rectangular boundary). Both fits restricted to
`[ConstantFit, PiecewiseLinearFit]` — only the piecewise-linear model has
an interpretable elbow, and that's what the decision-boundary overlay below
needs.

Each `adata` is dropped at the end of the function — these files are large
(4–7GB), and only the small derived arrays are kept for plotting.

In [5]:
from skimage.color import rgb2hed
from skimage.util import img_as_float

def compute_hed_tissue_signal(rgb_image):
    """Color-deconvolve an H&E RGB image into Hematoxylin/Eosin/DAB
    optical-density channels. Background has ~0 absorbance in both H and E,
    so H+E gives a cleaner tissue-vs-background signal than grayscale
    luminance. H alone is nuclei-specific.
    """
    rgb_float = img_as_float(rgb_image)  # handles uint8 0-255 or float 0-1 input
    hed = rgb2hed(rgb_float)
    h, e = hed[..., 0], hed[..., 1]
    tissue_signal = h + e
    return h, e, tissue_signal

In [6]:
def analyze_one(cfg):
    print(f"=== {cfg['name']} ===")
    adata = sc.read_h5ad(cfg["h5ad"])
    adata.obs["log1p_total_counts"] = np.log1p(np.asarray(adata.X.sum(axis=1)).ravel())

    sample_key = list(adata.uns["spatial"].keys())[0]
    scalefactors = adata.uns["spatial"][sample_key]["scalefactors"]
    hires_img = adata.uns["spatial"][sample_key]["images"]["hires"]
    microns_per_px = scalefactors["microns_per_pixel"] / scalefactors["tissue_hires_scalef"]
    bin_size_um = scalefactors["bin_size_um"]

    # tissue mask: multi-Otsu (background/tissue/nuclei), foreground = tissue+nuclei
    gray = np.asarray(Image.fromarray(hires_img).convert("L"))
    multi_thresh = threshold_multiotsu(gray, classes=3)
    region = np.digitize(gray, bins=multi_thresh)
    mask_fg = remove_small_holes(region < 2, max_size=MAX_HOLE_AREA)

    # align bins to the mask (hires-pixel space; obsm["spatial"] is (x,y) -> flip to (row,col))
    hires_xy = adata.obsm["spatial"] * scalefactors["tissue_hires_scalef"]
    hires_row = np.clip(hires_xy[:, 1].round().astype(int), 0, mask_fg.shape[0] - 1)
    hires_col = np.clip(hires_xy[:, 0].round().astype(int), 0, mask_fg.shape[1] - 1)
    is_tissue = mask_fg[hires_row, hires_col]

    # intrinsic: distance to background via the segmentation mask (unrestricted by capture area)
    coords_px = np.column_stack([hires_row[is_tissue], hires_col[is_tissue]])
    dist_seg_um = distance_to_mask(coords_px, ~mask_fg).to_numpy() * microns_per_px

    # extrinsic: distance to the capture-area's rectangular boundary (bin grid, not image)
    coords_grid_um = np.column_stack([
        adata.obs["array_row"].to_numpy()[is_tissue] * bin_size_um,
        adata.obs["array_col"].to_numpy()[is_tissue] * bin_size_um,
    ])
    dist_capture_um = distance_to_rectangular_border(coords_grid_um).to_numpy()

    scores = adata.obs.loc[is_tissue, ["log1p_total_counts"]].reset_index(drop=True)

    flow_seg = bosperrus.Flow.from_distances_and_scores(
        distances=pd.Series(dist_seg_um, name="distance_to_seg_mask_um"), scores=scores)
    flow_seg.flow(fits=[ConstantFit, PiecewiseLinearFit])

    flow_capture = bosperrus.Flow.from_distances_and_scores(
        distances=pd.Series(dist_capture_um, name="distance_to_capture_edge_um"), scores=scores)
    flow_capture.flow(fits=[ConstantFit, PiecewiseLinearFit])

    # full distance-to-background map (µm), for the elbow-contour overlay
    dist_seg_full_um = ndimage.distance_transform_edt(mask_fg) * microns_per_px

    result = {
        "name": cfg["name"],
        "gray": gray,
        "mask_fg": mask_fg,
        "flow_seg": flow_seg,
        "flow_capture": flow_capture,
        "dist_seg_full_um": dist_seg_full_um,
        "microns_per_px": microns_per_px,
        "hires_row_tissue": hires_row[is_tissue],
        "hires_col_tissue": hires_col[is_tissue],
        "log1p_total_counts_tissue": adata.obs.loc[is_tissue, "log1p_total_counts"].to_numpy(),
    }

    del adata
    gc.collect()
    return result

In [7]:
cfg = DATASETS[0]

In [8]:
adata = sc.read_h5ad(cfg["h5ad"])
adata.obs["log1p_total_counts"] = np.log1p(np.asarray(adata.X.sum(axis=1)).ravel())

In [9]:
adata

AnnData object with n_obs × n_vars = 1466755 × 18085
    obs: 'array_row', 'array_col', 'in_tissue', 'log1p_total_counts'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

In [ ]:
plt.scatter(adata.obs["array_row"], adata.obs["array_col"], c=adata.obs["in_tissue"], s=1)

In [ ]:
results = [analyze_one(cfg) for cfg in DATASETS]

## Summary comparison

Note on `observed_half_life`: it's deliberately scaled by `d_max` — it
reports what *fraction* of the maximum observed distance is affected, not
an absolute distance in µm. That's what makes it comparable across
datasets/tissue sizes of different extent; `elbow_um` (the fitted
`piecewise_linear_b` parameter itself) is the absolute-distance version.

In [ ]:
summary_rows = []
for r in results:
    for label, flow in [("intrinsic (seg. mask)", r["flow_seg"]), ("extrinsic (capture edge)", r["flow_capture"])]:
        fit = flow.best_fits["log1p_total_counts"]
        row = {
            "dataset": r["name"],
            "distance": label,
            "best_fit": fit.name,
            "effect_strength": flow.fit_quality.loc["observed_effect_strength", "log1p_total_counts"],
            # fraction of d_max affected, NOT a distance in microns -- see markdown above
            "half_life_frac_of_dmax": flow.fit_quality.loc["observed_half_life", "log1p_total_counts"],
            "n_tissue_bins": len(r["log1p_total_counts_tissue"]),
        }
        if isinstance(fit, PiecewiseLinearFit):
            row["elbow_um"] = fit.params["piecewise_linear_b"]
            row["slope_m"] = fit.params["piecewise_linear_m"]
        summary_rows.append(row)

pd.DataFrame(summary_rows)

In [ ]:
def plot_fit_curve(ax, fit, d):
    """Overlay the fitted curve (piecewise-linear, or a flat line if ConstantFit won) on a scatter axis."""
    d_line = np.linspace(0, d.max(), 200)
    if isinstance(fit, PiecewiseLinearFit):
        b, m, c = fit.params["piecewise_linear_b"], fit.params["piecewise_linear_m"], fit.params["piecewise_linear_c"]
        y_line = PiecewiseLinearFit.piecewise_plateau(d_line, b=b, m=m, c=c)
        ax.plot(d_line, y_line, color="tab:red", lw=2, label=f"piecewise-linear (elbow={b:.1f}µm)")
        ax.axvline(b, color="tab:red", lw=1, ls="--", alpha=0.7)
    else:
        ax.axhline(fit.params["constant_c"], color="tab:red", lw=2, label="constant (no effect)")
    ax.legend(fontsize=8, loc="best")

## Images

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
for ax, r in zip(axes, results):
    ax.imshow(r["gray"], cmap="gray")
    ax.set_title(r["name"])
    ax.axis("off")
plt.tight_layout()

## Segmentation masks

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
for ax, r in zip(axes, results):
    ax.imshow(r["mask_fg"], cmap="gray")
    ax.set_title(r["name"])
    ax.axis("off")
plt.tight_layout()

## Counts vs. distance to segmentation mask (intrinsic)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, r in zip(axes, results):
    d = r["flow_seg"].observations["distance_to_seg_mask_um"]
    s = r["flow_seg"].observations["log1p_total_counts"]
    ax.scatter(d, s, s=0.3, alpha=0.1, color="gray", rasterized=True)
    plot_fit_curve(ax, r["flow_seg"].best_fits["log1p_total_counts"], d)
    ax.set_xlabel("distance to segmentation mask background (μm)")
    ax.set_ylabel("log1p(total_counts)")
    ax.set_title(r["name"])
plt.tight_layout()

## Counts vs. distance to capture-area edge (extrinsic)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, r in zip(axes, results):
    d = r["flow_capture"].observations["distance_to_capture_edge_um"]
    s = r["flow_capture"].observations["log1p_total_counts"]
    ax.scatter(d, s, s=0.3, alpha=0.1, color="tab:purple", rasterized=True)
    plot_fit_curve(ax, r["flow_capture"].best_fits["log1p_total_counts"], d)
    ax.set_xlabel("distance to capture-area edge (μm)")
    ax.set_ylabel("log1p(total_counts)")
    ax.set_title(r["name"])
plt.tight_layout()

## Decision boundary overlay

Tissue image with a very transparent count overlay, plus both elbows drawn
as exclusion boundaries: the intrinsic (segmentation-mask) elbow as a
contour line on the full distance-to-background map, and the extrinsic
(capture-area) elbow as an inset rectangle (capture-area bounding box
shrunk inward by the elbow distance on each side). Bins inside either
boundary are the region a real analysis would need to exclude or correct.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, r in zip(axes, results):
    ax.imshow(r["gray"], cmap="gray")
    ax.scatter(r["hires_col_tissue"], r["hires_row_tissue"], c=r["log1p_total_counts_tissue"],
               cmap="viridis", s=0.3, alpha=0.05, rasterized=True)

    seg_fit = r["flow_seg"].best_fits["log1p_total_counts"]
    if isinstance(seg_fit, PiecewiseLinearFit):
        b_seg = seg_fit.params["piecewise_linear_b"]
        ax.contour(r["dist_seg_full_um"], levels=[b_seg], colors="tab:red", linewidths=1.5)

    capture_fit = r["flow_capture"].best_fits["log1p_total_counts"]
    if isinstance(capture_fit, PiecewiseLinearFit):
        b_capture_px = capture_fit.params["piecewise_linear_b"] / r["microns_per_px"]
        row0, row1 = r["hires_row_tissue"].min(), r["hires_row_tissue"].max()
        col0, col1 = r["hires_col_tissue"].min(), r["hires_col_tissue"].max()
        rect = Rectangle(
            (col0 + b_capture_px, row0 + b_capture_px),
            (col1 - col0) - 2 * b_capture_px, (row1 - row0) - 2 * b_capture_px,
            fill=False, edgecolor="tab:orange", linewidth=1.5, linestyle="--",
        )
        ax.add_patch(rect)

    ax.set_title(r["name"])
    ax.axis("off")

axes[0].plot([], [], color="tab:red", label="intrinsic elbow (seg. mask)")
axes[0].plot([], [], color="tab:orange", ls="--", label="extrinsic elbow (capture edge)")
axes[0].legend(fontsize=8, loc="lower left")
plt.tight_layout()

In [ ]:
from scipy import stats


def blade_style_sweep(distance_um, counts, bin_size_um=BIN_SIZE_UM, min_group_size=30):
    """BLADE-style ring-by-ring sweep: at each integer distance layer k,
    two-sample Welch t-test of counts at layer==k ("ring") vs layer>k
    ("remaining interior"). Walks outward from k=1. Returns the full
    p-value trajectory plus the smallest k where p >= 0.05 (the
    BLADE-style buffer), analogous to bosperrus's elbow.
    """
    layer = np.floor(np.asarray(distance_um) / bin_size_um).astype(int)
    counts = np.asarray(counts, dtype=float)
    valid = np.isfinite(counts) & np.isfinite(layer)
    max_layer = int(layer[valid].max()) if valid.any() else 0

    rows = []
    for k in range(1, max_layer + 1):
        ring = valid & (layer == k)
        interior = valid & (layer > k)
        if ring.sum() < min_group_size or interior.sum() < min_group_size:
            break
        p = stats.ttest_ind(counts[ring], counts[interior], equal_var=False).pvalue
        rows.append({
            "layer": k, "distance_um": k * bin_size_um, "p_value": p,
            "n_ring": int(ring.sum()), "n_interior": int(interior.sum()),
            "mean_ring": counts[ring].mean(), "mean_interior": counts[interior].mean(),
        })
    sweep_df = pd.DataFrame(rows)
    sig = sweep_df["p_value"] >= 0.05
    buffer_um = sweep_df.loc[sig, "distance_um"].min() if sig.any() else np.nan
    return sweep_df, buffer_um

In [ ]:
sweep_results = []
for cfg, res in zip(DATASETS, results):
    for dist_label, flow in [("intrinsic", res["flow_seg"]), ("extrinsic", res["flow_capture"])]:
        measure = flow._measures[0]
        d = flow.observations[flow._distance_key].to_numpy()
        raw = flow.observations[measure].to_numpy()
        corrected = flow.observations[f"BOSPERRUS corrected {measure}"].to_numpy()
        for counts_label, counts in [("raw", raw), ("bosperrus_corrected", corrected)]:
            sweep_df, buffer_um = blade_style_sweep(d, counts)
            sweep_results.append({
                "dataset": cfg["name"], "distance": dist_label, "counts": counts_label,
                "sweep_df": sweep_df, "buffer_um": buffer_um,
            })

comparison_table = pd.DataFrame([
    {"dataset": e["dataset"], "distance": e["distance"], "counts": e["counts"], "blade_buffer_um": e["buffer_um"]}
    for e in sweep_results
]).pivot_table(index=["dataset", "distance"], columns="counts", values="blade_buffer_um")
comparison_table

In [ ]:
fig, axes = plt.subplots(2, len(DATASETS), figsize=(5 * len(DATASETS), 7), squeeze=False)
colors = {"raw": "tab:blue", "bosperrus_corrected": "tab:orange"}

for col, cfg in enumerate(DATASETS):
    for row, dist_label in enumerate(["intrinsic", "extrinsic"]):
        ax = axes[row][col]
        for counts_label in ["raw", "bosperrus_corrected"]:
            entry = next(e for e in sweep_results
                         if e["dataset"] == cfg["name"] and e["distance"] == dist_label and e["counts"] == counts_label)
            df = entry["sweep_df"]
            if len(df):
                ax.plot(df["distance_um"], df["p_value"], marker="o", ms=3,
                        color=colors[counts_label], label=counts_label)
        ax.axhline(0.05, color="red", ls="--", lw=1)
        ax.set_yscale("log")
        ax.set_title(f"{cfg['name']} — {dist_label}")
        if row == 1:
            ax.set_xlabel("ring distance (µm)")
        if col == 0:
            ax.set_ylabel("t-test p-value")
        ax.legend(fontsize=7)

fig.tight_layout()